## 1. Import Libraries and Set Up the Environment

Import the required libraries and QuorumGR modules, set the random seed for reproducibility, and select the available device (CPU or GPU).

In [ ]:
from pathlib import Path
import sys
import torch


from pathlib import Path

# Set the path to your QuorumGR project folder
ROOT = Path(r"C:\path\to\your\QuorumGR")

# Check the project folder
print("Project folder:", ROOT)
print("Model file found:", (ROOT / "src" / "model.py").exists())

print("Project folder:", ROOT)
print("Model file found:", (ROOT / "src" / "model.py").exists())

# Check that the source folder exists
if not (ROOT / "src" / "model.py").exists():
    raise FileNotFoundError(
        f"QuorumGR source folder not found in {ROOT}. "
        "Start Jupyter from the project root or notebooks folder."
    )

print("Project folder:", ROOT)

# Add the src folder to Python's import path
sys.path.insert(0, str(ROOT / "src"))

# Import QuorumGR modules
from data import load_camra, validate_camra
from model import QuorumGR
from quorum import QuorumDecisionLayer

from train import (
    set_seed,
    train_user_preference,
    train_group_quality,
    split_indices,
)

from decision_train import train_collective_decision_rates

from clean_eval import (
    build_fixed_validation_candidates,
    evaluate_fixed_validation,
)

from test_eval import (
    build_official_group_test_candidates,
    evaluate_official_group_test,
    evaluate_official_group_test_tie_aware,
)

# Set random seed
seed = 42
set_seed(seed)

# Select CPU or GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project folder:", ROOT)
print("Device:", device)

## 2. Load the CAMRa2011 dataset

Place the seven CAMRa2011 text files in `data/CAMRa2011/` before running this cell.

In [11]:
data_dir = ROOT / "data" / "CAMRa2011"
data = load_camra(data_dir)
validate_camra(data)
print("Users:", data.num_users)
print("Groups:", data.num_groups)
print("Items:", data.num_items)

CAMRa2011 DATASET
Number of users: 602
Number of groups: 290
Number of items: 7710

User train ratings: 113334
User test ratings: 3010
Group train ratings: 143618
Group test ratings: 1450

User negative sets: 3010
Group negative sets: 1444

Group-test positives missing negative candidates: 0
User-test positives missing negative candidates: 0
Users: 602
Groups: 290
Items: 7710


## 3. Initialize the model

In [ ]:

# Use GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

model = QuorumGR(
    num_users=data.num_users,
    num_items=data.num_items,
    embedding_dim=64,
).to(device)

decision_layer = QuorumDecisionLayer(
    num_steps=5,
    dt=0.20,
    initial_commitment=0.10,
    discovery_strength=0.20,
    recruitment_strength=1.00,
    abandonment_strength=0.20,
    inhibition_strength=0.50,
    quorum_scale=2.0,
    quorum_temperature=0.25,
).to(device)

## 4. Train individual preferences (Stage A)

In [ ]:
# Stage A: Train individual user preferences
if not LOAD_SAVED_WEIGHTS:
    model = train_user_preference(
        model,
        data,
        device,
        epochs=100,
        batch_size=256,
        seed=seed,
    )
else:
    print("Stage A skipped. Using saved model.")

## 5. Train group quality (Stage B)

In [5]:
# Stage B: Train group-quality predictions
if not LOAD_SAVED_WEIGHTS:
    model = train_group_quality(
        model,
        data,
        device,
        epochs=75,
        batch_size=256,
        seed=seed,
    )
else:
    print("Stage B skipped. Using saved model.")

Group train / validation: 129257 / 14361
Epoch 001 | Train=0.922880 | Rating MSE=0.111044 | HR@10=0.924796 | NDCG@10=0.732396
Epoch 002 | Train=0.713142 | Rating MSE=0.111189 | HR@10=0.933570 | NDCG@10=0.754259
Epoch 003 | Train=0.676479 | Rating MSE=0.113291 | HR@10=0.936843 | NDCG@10=0.759770
Epoch 004 | Train=0.650840 | Rating MSE=0.114593 | HR@10=0.938375 | NDCG@10=0.767586
Epoch 005 | Train=0.637967 | Rating MSE=0.114518 | HR@10=0.938793 | NDCG@10=0.770517
Epoch 006 | Train=0.625077 | Rating MSE=0.110481 | HR@10=0.941369 | NDCG@10=0.774030
Epoch 007 | Train=0.616443 | Rating MSE=0.111488 | HR@10=0.941787 | NDCG@10=0.776615
Epoch 008 | Train=0.606857 | Rating MSE=0.114913 | HR@10=0.942901 | NDCG@10=0.779654
Epoch 009 | Train=0.598484 | Rating MSE=0.114547 | HR@10=0.943179 | NDCG@10=0.780901
Epoch 010 | Train=0.598443 | Rating MSE=0.116764 | HR@10=0.943597 | NDCG@10=0.782149
Epoch 011 | Train=0.586767 | Rating MSE=0.118563 | HR@10=0.943528 | NDCG@10=0.783149
Epoch 012 | Train=0.5811

## 6. Train the collective decision layer

This example trains all four decision rates: discovery, recruitment, abandonment, and inhibition (DRAI).

In [6]:
# Stage C: Train the collective decision rates
if not LOAD_SAVED_WEIGHTS:
    decision_layer = train_collective_decision_rates(
        model, decision_layer, data, device,
        epochs=80,
        batch_size=256,
        seed=seed,
    )
else:
    print("Decision-layer training skipped. Using saved weights.")

Decision train / validation: 129257 / 14361
Epoch 001 | Loss=1.048335 | HR@10=0.946661 | NDCG@10=0.790504 | D=0.0066 | R=3.5960 | A=1.8186 | I=1.7573
Epoch 002 | Loss=0.693929 | HR@10=0.947079 | NDCG@10=0.790933 | D=0.0042 | R=3.7343 | A=1.6739 | I=2.0143
Epoch 003 | Loss=0.689860 | HR@10=0.947009 | NDCG@10=0.790986 | D=0.0029 | R=3.8027 | A=1.4576 | I=2.4466
Epoch 004 | Loss=0.689797 | HR@10=0.946940 | NDCG@10=0.791005 | D=0.0019 | R=3.8733 | A=1.3053 | I=3.1591
Epoch 005 | Loss=0.686900 | HR@10=0.946870 | NDCG@10=0.790802 | D=0.0014 | R=3.9050 | A=1.1407 | I=4.1822
Epoch 006 | Loss=0.686919 | HR@10=0.946800 | NDCG@10=0.790606 | D=0.0010 | R=3.9590 | A=1.0273 | I=5.5570
Epoch 007 | Loss=0.682222 | HR@10=0.946800 | NDCG@10=0.790502 | D=0.0007 | R=4.0548 | A=0.9426 | I=7.2579
Epoch 008 | Loss=0.680658 | HR@10=0.946731 | NDCG@10=0.790131 | D=0.0006 | R=4.1716 | A=0.8468 | I=9.2289
Epoch 009 | Loss=0.676608 | HR@10=0.946452 | NDCG@10=0.789955 | D=0.0004 | R=4.2706 | A=0.7187 | I=11.4064
E

In [10]:
# Choose whether to train or load saved weights
LOAD_SAVED_WEIGHTS = False

if LOAD_SAVED_WEIGHTS:
    # Load previously trained model and decision layer
    model_path = ROOT / "checkpoints" / "camra_model.pt"
    decision_path = ROOT / "checkpoints" / "camra_decision.pt"

    model.load_state_dict(
        torch.load(model_path, map_location=device, weights_only=True)
    )

    decision_layer.load_state_dict(
        torch.load(decision_path, map_location=device, weights_only=True)
    )

    print("Trained model loaded successfully.")
else:
    print("Model initialized. Ready for training.")

Model initialized. Ready for training.


## 7. Evaluate on fixed validation candidates

In [7]:
# Use group-training data only to create the validation candidates.
_, val_indices = split_indices(
    len(data.group_train.ratings), val_fraction=0.10, seed=seed + 1
)
val_candidates = build_fixed_validation_candidates(
    data, val_indices, num_negatives=100, seed=seed
)

validation_results = evaluate_fixed_validation(
    model, decision_layer, data, val_candidates, device, batch_size=256
)
validation_results

{'quality': {'HR@5': 0.8981268713877864,
  'NDCG@5': 0.7847683021147498,
  'HR@10': 0.9474966924308892,
  'NDCG@10': 0.8008862095584395,
  'HR@20': 0.9780655943179445,
  'NDCG@20': 0.8087559568459473},
 'support': {'HR@5': 0.8383120952579904,
  'NDCG@5': 0.6948687445361189,
  'HR@10': 0.9233340296636724,
  'NDCG@10': 0.722642220940134,
  'HR@20': 0.9705452266555253,
  'NDCG@20': 0.7347408389241639},
 'commitment': {'HR@5': 0.8928347608105285,
  'NDCG@5': 0.7737957755384252,
  'HR@10': 0.9481233897360908,
  'NDCG@10': 0.7918567725206084,
  'HR@20': 0.977717429148388,
  'NDCG@20': 0.79943611193364},
 'quorum': {'HR@5': 0.8926258617087947,
  'NDCG@5': 0.7733478284498069,
  'HR@10': 0.947914490634357,
  'NDCG@10': 0.7914236092765006,
  'HR@20': 0.976255135436251,
  'NDCG@20': 0.7986710835744989}}

## 8. Evaluate on the official test candidates

Standard and tie-aware evaluation are shown separately. The supplied test functions report quality, support and commitment scores.

In [8]:
test_candidates = build_official_group_test_candidates(data)

test_results = evaluate_official_group_test(
    model, decision_layer, data, test_candidates, device, batch_size=256
)
tie_aware_results = evaluate_official_group_test_tie_aware(
    model, decision_layer, data, test_candidates, device, batch_size=256
)

print("Standard test:", test_results)
print("Tie-aware test:", tie_aware_results)

Official test rows: 1450
Evaluable rows: 1450
Missing negative sets: 0
Candidate-set sizes: [101]
Standard test: {'quality': {'HR@5': 0.6241379310344828, 'NDCG@5': 0.41693044728246226, 'HR@10': 0.8062068965517242, 'NDCG@10': 0.476549098573882, 'HR@20': 0.9213793103448276, 'NDCG@20': 0.5059285499309671}, 'support': {'HR@5': 0.6358620689655172, 'NDCG@5': 0.43329503026501887, 'HR@10': 0.8275862068965517, 'NDCG@10': 0.4958051799905711, 'HR@20': 0.9220689655172414, 'NDCG@20': 0.5199891241665544}, 'commitment': {'HR@5': 0.6303448275862069, 'NDCG@5': 0.4255053132155846, 'HR@10': 0.8165517241379311, 'NDCG@10': 0.4863045212318157, 'HR@20': 0.9248275862068965, 'NDCG@20': 0.5139644491261449}}
Tie-aware test: {'quality': {'HR@5': 0.6231034482758621, 'NDCG@5': 0.4158207860486261, 'HR@10': 0.8062068965517242, 'NDCG@10': 0.4757926783068427, 'HR@20': 0.9213793103448276, 'NDCG@20': 0.5051627376161772}, 'support': {'HR@5': 0.6358620689655172, 'NDCG@5': 0.4322619734139278, 'HR@10': 0.826551724137931, 'ND

## 9. Save trained weights and results

In [ ]:
import json

# Create folders for checkpoints and results
checkpoints_dir = ROOT / "checkpoints"
results_dir = ROOT / "results"

checkpoints_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

# Save trained weights only after training
if not LOAD_SAVED_WEIGHTS:
    torch.save(model.state_dict(), checkpoints_dir / "camra_model.pt")
    torch.save(decision_layer.state_dict(), checkpoints_dir / "camra_decision.pt")
    print("Trained model and decision layer saved.")
else:
    print("Existing checkpoints were not overwritten.")

# Save evaluation results
results = {
    "seed": seed,
    "validation": validation_results,
    "test": test_results,
    "test_tie_aware": tie_aware_results,
}

# Use a separate filename when evaluating loaded checkpoints
filename = (
    "camra_results_loaded.json"
    if LOAD_SAVED_WEIGHTS
    else "camra_results.json"
)

with open(results_dir / filename, "w") as f:
    json.dump(results, f, indent=2, default=float)

print("Results saved to:", results_dir / filename)

In [ ]:
# Check whether saved weights are available
model_path = ROOT / "checkpoints" / "camra_model.pt"
decision_path = ROOT / "checkpoints" / "camra_decision.pt"

print("Model checkpoint:", model_path.exists())
print("Decision checkpoint:", decision_path.exists())

## 9. Evaluate Trajectory Commitment and Adaptive Quorum

Evaluate the trajectory and quorum scores using the official
CAMRa2011 test candidates, including tie-aware ranking.

In [9]:
import torch
from test_eval import make_test_group_batch

@torch.no_grad()
def evaluate_extra_scores(
    model, decision_layer, data, test_candidates, device,
    batch_size=256, ks=(5, 10, 20)
):
    model.eval()
    decision_layer.eval()

    methods = ["trajectory", "quorum"]
    metrics = ["HR", "NDCG"]
    results = {
        mode: {
            name: {f"{metric}@{k}": 0.0
                   for k in ks for metric in metrics}
            for name in methods
        }
        for mode in ["standard", "tie_aware"]
    }

    # Group queries with the same number of candidates.
    by_size = {}
    for query in test_candidates:
        by_size.setdefault(len(query[3]), []).append(query)

    for queries in by_size.values():
        for start in range(0, len(queries), batch_size):
            batch = queries[start:start + batch_size]

            member_ids, member_mask, candidate_items = (
                make_test_group_batch(
                    batch, data, model.padding_user_id, device
                )
            )

            # Predict member support and group quality.
            output = model(
                member_ids, member_mask, candidate_items
            )

            # Run the collective decision process.
            decision = decision_layer(
                member_support=output["member_support"],
                member_mask=member_mask,
                collective_support=output["collective_support"],
                group_quality=output["group_quality"],
            )

            scores = {
                "trajectory": decision["trajectory_commitment"],
                "quorum": decision["quorum_score"],
            }

            for name, values in scores.items():
                # The positive item is in column zero.
                positive = values[:, :1]

                # Standard evaluation: ties favor the positive item.
                greater = (values[:, 1:] > positive).sum(dim=1)
                rank = greater + 1

                # Tie-aware evaluation: average over possible tied ranks.
                tied = (values[:, 1:] == positive).sum(dim=1)
                tie_size = tied + 1

                for k in ks:
                    hits = rank <= k

                    hr = hits.float()
                    ndcg = torch.where(
                        hits,
                        1.0 / torch.log2(rank.float() + 1),
                        torch.zeros_like(rank, dtype=torch.float32),
                    )

                    results["standard"][name][f"HR@{k}"] += (
                        hr.sum().item()
                    )
                    results["standard"][name][f"NDCG@{k}"] += (
                        ndcg.sum().item()
                    )

                    # Expected values under random tie-breaking.
                    positions = torch.arange(
                        1, k + 1, device=device
                    ).unsqueeze(0)

                    inside_tie = (
                        (positions >= rank.unsqueeze(1))
                        & (positions < (rank + tie_size).unsqueeze(1))
                    )

                    expected_hr = (
                        inside_tie.float().sum(dim=1)
                        / tie_size.float()
                    )

                    discounts = 1.0 / torch.log2(
                        positions.float() + 1
                    )
                    expected_ndcg = (
                        (inside_tie.float() * discounts).sum(dim=1)
                        / tie_size.float()
                    )

                    results["tie_aware"][name][f"HR@{k}"] += (
                        expected_hr.sum().item()
                    )
                    results["tie_aware"][name][f"NDCG@{k}"] += (
                        expected_ndcg.sum().item()
                    )

    # Average metrics over all official test queries.
    total = len(test_candidates)
    if total == 0:
        raise ValueError("No evaluable test candidates found.")

    for mode in results:
        for name in methods:
            for metric in results[mode][name]:
                results[mode][name][metric] /= total

    return results


# Reuse the trained model and official test candidates.
extra_results = evaluate_extra_scores(
    model,
    decision_layer,
    data,
    test_candidates,
    device,
)

print("Standard test:")
print(extra_results["standard"])

print("\nTie-aware test:")
print(extra_results["tie_aware"])

Standard test:
{'trajectory': {'HR@5': 0.6275862068965518, 'NDCG@5': 0.42543544900828395, 'HR@10': 0.8186206896551724, 'NDCG@10': 0.4878901198814655, 'HR@20': 0.9255172413793104, 'NDCG@20': 0.5152026577653556}, 'quorum': {'HR@5': 0.6303448275862069, 'NDCG@5': 0.4255053132155846, 'HR@10': 0.8165517241379311, 'NDCG@10': 0.4863045212318157, 'HR@20': 0.9262068965517242, 'NDCG@20': 0.5142711297396956}}

Tie-aware test:
{'trajectory': {'HR@5': 0.6268965517241379, 'NDCG@5': 0.4244352827400997, 'HR@10': 0.8186206896551724, 'NDCG@10': 0.4871086488921067, 'HR@20': 0.9255172413793104, 'NDCG@20': 0.5144155620706492}, 'quorum': {'HR@5': 0.6296551724137931, 'NDCG@5': 0.42450513642409754, 'HR@10': 0.8165517241379311, 'NDCG@10': 0.4855230712890625, 'HR@20': 0.9262068965517242, 'NDCG@20': 0.5134840550915948}}
